In [ ]:
!pip install beautifulsoup4 requests google-genai pandas

In [ ]:
import getpass
import requests
from bs4 import BeautifulSoup
import pandas as pd
from google import genai

# 1. Securely Input Gemini API Key
api_key = getpass.getpass("Paste your Google Gemini API Key: ")
client = genai.Client(api_key=api_key)

# 2. Define Competitor Target URLs (Changelogs / Product Blogs)
# For demonstration, we use live tech changelog feeds or web endpoints
competitor_urls = {
    "Competitor A (Linear Release Notes)": "https://linear.app/readme",
    "Competitor B (GitHub Changelog Sample)": "https://github.blog/changelog/"
}

scraped_intel = []

print("1/3: Scraping competitor changelogs and web updates...")

# Header to mimic standard browser request during scraping
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

for name, url in competitor_urls.items():
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            # Extract text content from main article or changelog body tags
            paragraphs = soup.find_all(['p', 'h2', 'h3', 'li'])
            text_content = " ".join([p.get_text().strip() for p in paragraphs[:25]])

            scraped_intel.append(f"=== {name} ({url}) ===\n{text_content}\n")
            print(f" Successfully scraped: {name}")
        else:
            print(f"⚠️ Failed to reach {name} (HTTP {response.status_code})")
    except Exception as e:
        print(f"❌ Error scraping {name}: {str(e)}")

# Combine all scraped raw content into a single payload
consolidated_scraped_payload = "\n".join(scraped_intel)

# 3. LLM Summarization Engine (Gemini API)
print("\n2/3: Passing scraped web data to Gemini for strategic synthesis...")

prompt = f"""
You are a Vice President of Product Strategy. Below is raw scraped web data from recent competitor changelogs and release notes:

{consolidated_scraped_payload[:8000]}  # Truncate safely to fit context window

Process this scraped market data and produce a executive-level "Monthly Competitive Intelligence & Roadmap Memo". Format strictly in clean Markdown with these sections:

1. 📌 Executive Strategic Summary
Summarize the major strategic product shifts observed across these competitor updates in 2 sentences.

2. ⚔️ Competitor Release Matrix
Format a Markdown table with columns:
| Competitor | Major Shipped Capability | Targeted User Segment | Strategic Impact |

3. 🔍 Identified Feature Gaps & Market Threats
Detail 2 critical capabilities competitors have recently introduced that pose a threat to our market share.

4. 🛡️ Strategic Implications & Roadmap Counter-Strategies
Provide 2 concrete, prioritized recommendations for our own product roadmap to counter these movements or exploit competitor blind spots.
"""

response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=prompt
)

print("\n3/3: Execution Complete! Generated Monthly Executive Memo:\n")
print("="*80)
print(response.text)
print("="*80)

Paste your Google Gemini API Key: ··········
1/3: Scraping competitor changelogs and web updates...
 Successfully scraped: Competitor A (Linear Release Notes)
 Successfully scraped: Competitor B (GitHub Changelog Sample)

2/3: Passing scraped web data to Gemini for strategic synthesis...

3/3: Execution Complete! Generated Monthly Executive Memo:

# Monthly Competitive Intelligence & Roadmap Memo

### 📌 Executive Strategic Summary
Competitors are bifurcating their market strategies, with Linear doubling down on high-fidelity, brand-led user experiences to capture premium developer mindshare, while GitHub aggressively matures its enterprise-grade governance, AI integrations (Copilot), and security compliance. To protect our market share, we must balance a refined focus on product craftsmanship with robust, enterprise-ready platform administration capabilities.

---

### ⚔️ Competitor Release Matrix

| Competitor | Major Shipped Capability | Targeted User Segment | Strategic Impact |
| :